In [1]:
from Test_prep_MATLAB import Prep
from evaluation_script_translation import Test

c:\Code_Playground_offline\low_res_face_rec\faceMOE_implementation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Prep()


Extracting on: CUDA
Loading Swin Backbone...


Loading weights: 100%|██████████| 423/423 [00:00<00:00, 7658.99it/s]


Loading trained MoE weights...
Extracting features for 4443 images in ./Dataset/tinyface/Testing_Set/Gallery_Match...


Processing Gallery_Match: 100%|██████████| 4443/4443 [06:14<00:00, 11.85img/s]


Extracting features for 3728 images in ./Dataset/tinyface/Testing_Set/Probe...


Processing Probe: 100%|██████████| 3728/3728 [05:26<00:00, 11.41img/s]


Extracting features for 153428 images in ./Dataset/tinyface/Testing_Set/Gallery_Distractor...


Processing Gallery_Distractor: 100%|██████████| 153428/153428 [3:50:34<00:00, 11.09img/s]  


Saving matrices for MATLAB...
Done! You can now run 'test_face_identification.m' in MATLAB!


In [2]:
Test()

Loading feature matrices and ID pairs...
Constructing Gallery...
Calculating Euclidean Distances (pdist2 equivalent)...
Scoring probe images...


Scoring Probes: 100%|██████████| 3728/3728 [13:38<00:00,  4.56probe/s]



🏆 EVALUATION RESULTS 🏆
mAP (Mean Average Precision): 1.02%
Rank-1 Accuracy:  2.17%
Rank-5 Accuracy:  4.27%
Rank-10 Accuracy: 5.23%


In [21]:
from config import Gallery_ids_mat
import os
gallery_dir = "./Dataset/tinyface/Testing_Set/Gallery_Match"
probe_dir = "./Dataset/tinyface/Testing_Set/Probe"

files = sorted(os.listdir(gallery_dir))
print(files[:15])

['1000_15.jpg', '1004_46.jpg', '1004_72.jpg', '1004_91.jpg', '1005_16.jpg', '1005_2.jpg', '1006_15.jpg', '1007_74.jpg', '1007_75.jpg', '1007_8.jpg', '1008_1.jpg', '1008_5.jpg', '1015_12.jpg', '1015_13.jpg', '1015_5.jpg']


In [5]:
import re
def numeric_key(fname):
    return int(re.search(r'\d+', fname).group())

image_files = sorted(
    [f for f in os.listdir(gallery_dir) if f.endswith(('.jpg', '.png'))],
    key=numeric_key
)
image_files[:10]

['1_106.jpg',
 '1_118.jpg',
 '1_128.jpg',
 '1_158.jpg',
 '1_160.jpg',
 '1_189.jpg',
 '1_64.jpg',
 '1_87.jpg',
 '1_93.jpg',
 '2_9.jpg']

In [9]:
import re

def numeric_key(fname):
    # Matches patterns like "1_106.jpg" -> (1, 106)
    match = re.match(r'(\d+)_(\d+)', fname)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    # Fallback in case some filenames don't match the pattern
    return (float('inf'), fname)

image_files = sorted(
    [f for f in os.listdir(gallery_dir) if f.endswith(('.jpg', '.png'))],
    key=numeric_key
)
image_files[:30]

['1_64.jpg',
 '1_87.jpg',
 '1_93.jpg',
 '1_106.jpg',
 '1_118.jpg',
 '1_128.jpg',
 '1_158.jpg',
 '1_160.jpg',
 '1_189.jpg',
 '2_9.jpg',
 '5_22.jpg',
 '5_24.jpg',
 '6_9.jpg',
 '6_10.jpg',
 '7_47.jpg',
 '9_19.jpg',
 '9_27.jpg',
 '11_20.jpg',
 '11_30.jpg',
 '11_31.jpg',
 '14_6.jpg',
 '19_18.jpg',
 '20_10.jpg',
 '20_43.jpg',
 '20_104.jpg',
 '20_129.jpg',
 '20_146.jpg',
 '20_156.jpg',
 '20_175.jpg',
 '20_183.jpg']

In [15]:
import scipy.io as sio
import config
gallery_pairs = sio.loadmat(config.Gallery_ids_mat)
probe_pairs = sio.loadmat(config.Probe_ids_mat)
print(gallery_pairs.keys())  # find the actual field names first
print(probe_pairs.keys())  # find the actual field names first

dict_keys(['__header__', '__version__', '__globals__', 'gallery_ids', 'gallery_set'])
dict_keys(['__header__', '__version__', '__globals__', 'probe_ids', 'probe_set'])


In [18]:
import scipy.io as sio

data = sio.loadmat(config.Gallery_ids_mat)
data2 = sio.loadmat(config.Probe_ids_mat)
print(type(data['gallery_set']), data['gallery_set'].shape, data['gallery_set'].dtype)
print(type(data2['probe_set']), data2['probe_set'].shape, data2['probe_set'].dtype)
# print(type(data['gallery_ids']), data['gallery_ids'].shape)

# Peek at the first few raw entries before assuming structure
print(data['gallery_set'][:5])
print(data['gallery_ids'][:5])
print(data2['probe_set'][:5])
print(data2['probe_ids'][:5])

<class 'numpy.ndarray'> (4443, 1) object
<class 'numpy.ndarray'> (3728, 1) object
[[array(['1_106.jpg'], dtype='<U9')]
 [array(['1_118.jpg'], dtype='<U9')]
 [array(['1_128.jpg'], dtype='<U9')]
 [array(['1_158.jpg'], dtype='<U9')]
 [array(['1_160.jpg'], dtype='<U9')]]
[[1]
 [1]
 [1]
 [1]
 [1]]
[[array(['1_162.jpg'], dtype='<U9')]
 [array(['1_184.jpg'], dtype='<U9')]
 [array(['1_185.jpg'], dtype='<U9')]
 [array(['1_24.jpg'], dtype='<U8')]
 [array(['1_44.jpg'], dtype='<U8')]]
[[1]
 [1]
 [1]
 [1]
 [1]]


In [19]:
gallery_img_names = [str(x[0][0]) for x in data['gallery_set']]
probe_img_names = [str(x[0][0]) for x in data2['probe_set']]
print(len(gallery_img_names))
print(len(probe_img_names))
print(gallery_img_names[:10])
print(probe_img_names[:10])
print(data2['probe_ids'].flatten()[:10])  

4443
3728
['1_106.jpg', '1_118.jpg', '1_128.jpg', '1_158.jpg', '1_160.jpg', '1_189.jpg', '1_64.jpg', '1_87.jpg', '1_93.jpg', '2_9.jpg']
['1_162.jpg', '1_184.jpg', '1_185.jpg', '1_24.jpg', '1_44.jpg', '1_65.jpg', '1_69.jpg', '1_76.jpg', '1_88.jpg', '2_11.jpg']
[1 1 1 1 1 1 1 1 1 2]


In [22]:
official_order = probe_img_names  # from the .mat file, full list
python_sorted = sorted([f for f in os.listdir(probe_dir) if f.endswith(('.jpg','.png'))])

print(len(official_order), len(python_sorted))
print(official_order == python_sorted)  # should be True

# If False, find exactly where they diverge:
for i, (a, b) in enumerate(zip(official_order, python_sorted)):
    if a != b:
        print(f"First mismatch at index {i}: official={a}, python_sorted={b}")
        break

3728 3728
False
First mismatch at index 0: official=1_162.jpg, python_sorted=1000_9.jpg
